# Instacart Market Basket Analysis
**Dataset**: psparks/instacart-market-basket-analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

BASE = 'shardul/instacart-market-basket'


## 1. Load Data

In [ ]:
aisles = pd.read_csv(f'{BASE}/aisles.csv')
departments = pd.read_csv(f'{BASE}/departments.csv')
products = pd.read_csv(f'{BASE}/products.csv')
orders = pd.read_csv(f'{BASE}/orders.csv')
op_train = pd.read_csv(f'{BASE}/order_products__train.csv')
op_prior = pd.read_csv(f'{BASE}/order_products__prior.csv')

print('Loaded all files:')
for name, df in [('aisles', aisles), ('departments', departments), ('products', products), ('orders', orders), ('op_train', op_train), ('op_prior', op_prior)]:
    print(f'  {name}: {df.shape[0]:,} rows x {df.shape[1]} cols')


## 2. Schema & Data Quality

In [ ]:
for name, df in [('aisles', aisles), ('departments', departments), ('products', products), ('orders', orders), ('op_train', op_train), ('op_prior', op_prior)]:
    print(f'\n=== {name} ===')
    print(df.dtypes)
    print(f'Nulls:\n{df.isnull().sum()}')
    print(f'Unique: {df.nunique().to_dict()}')


## 3. Key Statistics

In [ ]:
print(f"Total unique users: {orders['user_id'].nunique():,}")
print(f"Total orders: {orders['order_id'].nunique():,}")
print(f"Total products: {products['product_id'].nunique():,}")
print(f"Total aisles: {aisles['aisle_id'].nunique():,}")
print(f"Total departments: {departments['department_id'].nunique():,}")
print(f"Prior order-product pairs: {len(op_prior):,}")
print(f"Train order-product pairs: {len(op_train):,}")


## 4. Order Behavior Analysis

In [ ]:
# Orders per user
opu = orders.groupby('user_id').size()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(opu.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Orders per User')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Number of Users')
axes[0].axvline(opu.mean(), color='red', linestyle='--', label=f'Mean: {opu.mean():.1f}')
axes[0].legend()

# Day of week
dow_counts = orders['order_dow'].value_counts().sort_index()
day_names = ['Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri']
axes[1].bar(range(7), dow_counts.values, color=sns.color_palette('viridis', 7))
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_names)
axes[1].set_title('Orders by Day of Week')
axes[1].set_ylabel('Number of Orders')
plt.tight_layout()
plt.show()
print(f"Orders per user — Mean: {opu.mean():.1f}, Median: {opu.median():.1f}, Min: {opu.min()}, Max: {opu.max()}")


In [ ]:
# Hour of day distribution
fig, ax = plt.subplots(figsize=(14, 5))
hod = orders['order_hour_of_day'].value_counts().sort_index()
ax.bar(hod.index, hod.values, color=sns.color_palette('coolwarm', 24))
ax.set_title('Orders by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Number of Orders')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

# Peak hours
peak = hod.nlargest(3)
print(f"Peak ordering hours: {peak.index.tolist()}")


In [ ]:
# Days since prior order
dspo = orders['days_since_prior_order'].dropna()
fig, ax = plt.subplots(figsize=(14, 5))
ax.hist(dspo.values, bins=31, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_title('Days Since Prior Order')
ax.set_xlabel('Days')
ax.set_ylabel('Frequency')
ax.axvline(dspo.mean(), color='red', linestyle='--', label=f'Mean: {dspo.mean():.1f} days')
ax.axvline(dspo.median(), color='orange', linestyle='--', label=f'Median: {dspo.median():.1f} days')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Mode: {dspo.mode().values[0]:.0f} days (likely weekly shoppers)")


## 5. Product & Department Analysis

In [ ]:
# Merge products with departments and aisles
prod_full = products.merge(aisles, on='aisle_id').merge(departments, on='department_id')

# Products per department
fig, ax = plt.subplots(figsize=(14, 6))
dept_counts = prod_full['department'].value_counts()
dept_counts.plot(kind='barh', ax=ax, color=sns.color_palette('Set2', len(dept_counts)))
ax.set_title('Number of Products per Department')
ax.set_xlabel('Product Count')
plt.tight_layout()
plt.show()


In [ ]:
# Top 20 aisles by product count
fig, ax = plt.subplots(figsize=(14, 8))
aisle_counts = prod_full['aisle'].value_counts().head(20)
aisle_counts.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 20 Aisles by Product Count')
ax.set_xlabel('Product Count')
plt.tight_layout()
plt.show()


## 6. Most Ordered Products

In [ ]:
# Most ordered products from prior orders
prod_orders = op_prior.groupby('product_id').size().reset_index(name='order_count')
prod_orders = prod_orders.merge(prod_full, on='product_id').sort_values('order_count', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
top20 = prod_orders.head(20)
ax.barh(range(20), top20['order_count'].values, color=sns.color_palette('magma', 20))
ax.set_yticks(range(20))
ax.set_yticklabels(top20['product_name'].values)
ax.set_xlabel('Number of Orders')
ax.set_title('Top 20 Most Ordered Products')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 7. Reorder Behavior

In [ ]:
# Overall reorder rates
print(f"Reorder rate (prior): {op_prior['reordered'].mean()*100:.1f}%")
print(f"Reorder rate (train): {op_train['reordered'].mean()*100:.1f}%")

# Reorder rate by department
prod_reorder = op_prior.merge(prod_full[['product_id','department','aisle']], on='product_id')
dept_reorder = prod_reorder.groupby('department')['reordered'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(dept_reorder)))
dept_reorder.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Reorder Rate by Department')
ax.set_xlabel('Reorder Rate')
ax.axvline(op_prior['reordered'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall avg')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Basket Size Analysis

In [ ]:
basket_prior = op_prior.groupby('order_id').size()
basket_train = op_train.groupby('order_id').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(basket_prior.values, bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[0].set_title(f'Basket Size (Prior) — Mean: {basket_prior.mean():.1f}')
axes[0].set_xlabel('Items per Order')
axes[0].set_ylabel('Frequency')
axes[0].set_xlim(0, 60)

axes[1].hist(basket_train.values, bins=50, edgecolor='black', alpha=0.7, color='salmon')
axes[1].set_title(f'Basket Size (Train) — Mean: {basket_train.mean():.1f}')
axes[1].set_xlabel('Items per Order')
axes[1].set_xlim(0, 60)
plt.tight_layout()
plt.show()


## 9. Add-to-Cart Order — First Items Added

In [ ]:
first_adds = op_prior[op_prior['add_to_cart_order'] == 1].merge(prod_full, on='product_id')
first_add_top = first_adds['product_name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 6))
first_add_top.plot(kind='barh', ax=ax, color='goldenrod')
ax.set_title('Top 15 Products Added FIRST to Cart (Planned Purchases)')
ax.set_xlabel('Frequency')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 10. Department Co-occurrence in Baskets

In [ ]:
# Sample baskets for co-occurrence (full data too large)
sample_orders = op_prior.merge(prod_full[['product_id','department']], on='product_id')
sample_ids = sample_orders['order_id'].drop_duplicates().sample(50000, random_state=42)
sample = sample_orders[sample_orders['order_id'].isin(sample_ids)]

# Build co-occurrence matrix
from itertools import combinations
dept_pairs = Counter()
for oid, grp in sample.groupby('order_id')['department']:
    depts = grp.unique()
    for a, b in combinations(sorted(depts), 2):
        dept_pairs[(a, b)] += 1

# Create matrix
all_depts = sorted(departments['department'].unique())
cooc = pd.DataFrame(0, index=all_depts, columns=all_depts)
for (a, b), cnt in dept_pairs.items():
    cooc.loc[a, b] = cnt
    cooc.loc[b, a] = cnt

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cooc, cmap='YlOrRd', ax=ax, fmt=',')
ax.set_title('Department Co-occurrence in Baskets (50k sample)')
plt.tight_layout()
plt.show()


## 11. Eval Set Breakdown

In [ ]:
orders['eval_set'].value_counts().plot(kind='bar', color=['steelblue','coral','seagreen'], edgecolor='black')
plt.title('Eval Set Distribution')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()


## 12. Relevance to Shelf Optimization / Planogram AI

**Strengths:**
- SKU Performance Scoring: Order frequency + reorder rate = velocity proxy
- Product Affinity Modeling: Co-purchase patterns from basket data
- Department/Aisle adjacency: Which departments appear together in baskets
- Demand forecasting: Order timing patterns (dow, hour, days_since_prior)
- Basket composition: What gets bought together, add-to-cart sequence

**Limitations:**
- No price/cost data — cannot compute margin-based scores
- No physical shelf/store layout data
- No inventory/stockout data
- User IDs are anonymized — no demographics
